### Pré-processamento do BETA dataset

Este notebook organiza os dados do participante S001 no formato do MNE, `(n_epochs, n_channels, n_times)`, cria os objetos Epochs, aplica os filtros temporais e salva os resultados para o notebook de visualização.

In [15]:
# Carregar os dados

import numpy as np

data = np.load('../data/output/S001_pre_pros.npy')

print(data.dtype) # float64
print(data.shape)

channels_index, time_points, electrode_index, block_index, target_index = data.shape
channels = ['POz', 'PO3', 'PO4', 'PO5', 'PO6', 'Oz', 'O1', 'O2']
fs = 250

float64
(8, 710, 2, 10, 12)


**Monta o array data `[canal, tempo, eletrodo, bloco, alvo]` na forma esperada pelo MNE [n_epochs, n_channels, n_times]**

Para isso, cada par (bloco, alvo) corresponde a uma epoch no formato [8, 710]. Como cada conjunto tem 10 blocos e 12 alvos, totalizam 120 epochs. Os tipos de eletrodos `dry` e `wet` foram separados em dois conjuntos, por serem duas toucas diferentes. Por fim, os labels guardam o índice do alvo de 0 a 11 de cada epoch.

In [16]:
import mne

# Move a ordem do array e faz um reshape

epochs_all = data.transpose(2, 3, 4, 0, 1)
print(epochs_all.shape) # [eletrodo, bloco, alvo, canal, tempo]

epochs_all = epochs_all.reshape(electrode_index * block_index * target_index, channels_index, time_points)
print(epochs_all.shape)

(2, 10, 12, 8, 710)
(240, 8, 710)


In [17]:
# Separar entre touca com eletrodos molhados e secos

number_of_epochs_per_electrode = block_index * target_index

epochs_dry = epochs_all[0:number_of_epochs_per_electrode]
epochs_wet = epochs_all[number_of_epochs_per_electrode:]

print(epochs_dry.shape)
print(epochs_wet.shape)

(120, 8, 710)
(120, 8, 710)


**Verificação**

Como cada época deve ser idêntica ao trecho correspondente do array original, o trecho abaixo compara a época 0 (bloco 0, alvo 0) e a época 13 (bloco 1, alvo 1). Como os loops percorrem os blocos e, dentro de cada bloco, os alvos, a época de índice `b * 12 + t` corresponde a `(bloco b, alvo t)`.

In [18]:
target_labels = np.tile(np.arange(target_index), block_index)

print(np.array_equal(epochs_dry[0], data[:, :, 0, 0, 0]))
print(np.array_equal(epochs_dry[13], data[:, :, 0, 1, 1]))
print(np.array_equal(epochs_wet[13], data[:, :, 1, 1, 1]))
print(target_labels[0:24])
print(target_labels.shape)

True
True
True
[ 0  1  2  3  4  5  6  7  8  9 10 11  0  1  2  3  4  5  6  7  8  9 10 11]
(120,)


**Objeto info**

In [19]:
sfreq = fs
ch_names = channels

info = mne.create_info(ch_names, sfreq=sfreq, ch_types='eeg')
info

<Info | 7 non-empty values
 bads: []
 ch_names: POz, PO3, PO4, PO5, PO6, Oz, O1, O2
 chs: 8 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 125.0 Hz
 meas_date: unspecified
 nchan: 8
 projs: []
 sfreq: 250.0 Hz
>

### Criação do EpochsArray

Com o info pronto, foram criados os eventos, que dizem ao MNE qual alvo foi olhado em cada epoch. Como o MNE assume que o sinal está em volts e o BETA está em microvolts, os dados foram multiplicados por 1 × 10⁻⁶ ao criar o objeto. 

O tmin foi definido como -0,5 s para que o zero do eixo de tempo marque o início do estímulo. Foram também separados dois objetos para cada tipo de eletrodo (dry ou wet), cada um com 120 epochs.

In [20]:
pre_stimulus_time = 0.5

# Eventos: índice da epoch, valor anterior e código do alvo (os alvos foram separados em códigos para poder selecionar as epochs por nome nas etapas seguintes).
events = np.column_stack((
    np.arange(len(target_labels)),
    np.zeros(len(target_labels), dtype=int),
    target_labels
))

# Nome de cada código de alvo
event_id = {}
for target in range(target_index):
    event_id['alvo_' + str(target)] = target

# Objetos Epochs em volts, com tempo zero no início do estímulo
mne_dry = mne.EpochsArray(epochs_dry * 1e-6, info, events, tmin=-pre_stimulus_time, event_id=event_id)
mne_wet = mne.EpochsArray(epochs_wet * 1e-6, info, events, tmin=-pre_stimulus_time, event_id=event_id)

print(mne_dry)
print(mne_wet)

Not setting metadata
120 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
120 matching events found
No baseline correction applied
0 projection items activated
<EpochsArray | 120 events (all good), -0.5 – 2.336 s (baseline off), ~5.2 MiB, data loaded,
 'alvo_0': 10
 'alvo_1': 10
 'alvo_2': 10
 'alvo_3': 10
 'alvo_4': 10
 'alvo_5': 10
 'alvo_6': 10
 'alvo_7': 10
 'alvo_8': 10
 'alvo_9': 10
 and 2 more events ...>
<EpochsArray | 120 events (all good), -0.5 – 2.336 s (baseline off), ~5.2 MiB, data loaded,
 'alvo_0': 10
 'alvo_1': 10
 'alvo_2': 10
 'alvo_3': 10
 'alvo_4': 10
 'alvo_5': 10
 'alvo_6': 10
 'alvo_7': 10
 'alvo_8': 10
 'alvo_9': 10
 and 2 more events ...>


### Definição e aplicação de filtros

In [ ]:
# Definições de filtros

# Filtro passa-banda já foi aplicado 
def butter_bandpass(data, lowcut, highcut, fs=512, order=4):
    nyq = fs * 0.5
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='bandpass')
    return signal.filtfilt(b, a, data)


def butter_lowpass(data, lowcut, fs=512, order=4):
    nyq = fs * 0.5
    low = lowcut / nyq
    b, a = signal.butter(order, low, btype='lowpass')
    return signal.filtfilt(b, a, data)


def butter_highpass(data, highcut, fs=512, order=4):
    nyq = fs * 0.5
    high = highcut / nyq
    b, a = signal.butter(order, high, btype='highpass')
    return signal.filtfilt(b, a, data)

# Filtro notch já foi aplicado para remover ruído (50Hz) da power-line
def butter_notch(data, cutoff, var=1, fs=512, order=4):
    nyq = fs * 0.5
    low = (cutoff - var) / nyq
    high = (cutoff + var) / nyq
    b, a = signal.iirfilter(order, [low, high], btype='bandstop', ftype="butter")
    return signal.filtfilt(b, a, data)